# Modul 10 · Unüberwachtes Lernen
**Data Science 1** · linkit courses

Welche Gruppen und Ausreißer stecken in Daten ohne Zielgröße? Du prüfst die Zyklen der Messbox an den Fräszentren M-01 und M-02 auf Lücken und Messfehler, rechnest Abstände und einen Lloyd-Schritt von Hand, wählst die Zahl der Gruppen mit der Silhouette, liest Profile in Werkseinheiten, prüfst die Stabilität, clusterst Maschinen hierarchisch, suchst dichte Gruppen mit DBSCAN, verdichtest mit PCA, findest Ausreißer je Gruppe und prüfst am Ende mit den Programmen laut Leitstand. Anders als im Skript nimmst du alle fünf Merkmale der Messbox, also auch die Spitzenleistung.

**Lernziele**
1. Ähnlichkeit über Abstände messen und Merkmale passend skalieren
2. k-Means anwenden und die Zahl der Gruppen begründet wählen
3. Dendrogramme lesen und Verknüpfungen hierarchischer Verfahren vergleichen
4. Mit DBSCAN dichte Gruppen und Rauschen trennen
5. Mit PCA Daten verdichten, Varianzanteile und Ladungen deuten
6. Ausreißer mit Isolation Forest und robusten Abständen finden
7. Gruppen ohne Zielgröße prüfen: Stabilität, Profile, wenige Etiketten

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import von scikit-learn lädt im Browser SciPy und scikit-learn, das dauert einmalig etwa 10 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt. Wo ein späteres TODO ein früheres Ergebnis braucht, springt eine Vorgabe-Zelle ein, solange das frühere offen ist.
- Mindestziel sind TODO 1 bis 13. Die beiden Bonusaufgaben vertiefen.
- Aller Zufall hängt an `SEED` in der Zelle nach den Importen: `random_state=SEED`, für Bootstrap `np.random.default_rng(SEED + 1)`. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg auch mit einem anderen Seed besteht.
- Die Nummern der Gruppen sind willkürlich. Kein Selbsttest fragt nach einer festen Nummer, sie vergleichen Gruppen über Profil, Größe oder den bereinigten Rand-Index.
- Gib Ergebnisse mit `float(...)`, `round(...)` oder der Hilfsfunktion `zeige` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["zyklen.csv", "stoerungen.csv", "sensordaten.csv", "produktion.csv", "maschinen.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Ähnlich heißt nah
*Lerneinheit 1 · Skript Abschnitt 1*

Jeder Zyklus ist ein Punkt im Merkmalsraum, ähnlich heißt nah. Der euklidische Abstand $d(x, y) = \sqrt{\sum_j (x_j - y_j)^2}$ zählt eine Sekunde so viel wie eine Kilowattstunde, deshalb gehören die Merkmale vorher skaliert, etwa standardisiert zu $z = (x - \bar x)/s$. Abstandsverfahren brauchen in jeder Zeile jeden Wert, und Werte, die physikalisch nicht sein können, gehören vorher geprüft.

Die nächste Zelle lädt die Zyklen wie im Skript. `FUENF` nennt die fünf Merkmale der Messbox, mit denen du clusterst, `VIER` die vier aus dem Skript ohne Spitzenleistung. Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen, Tabellen mit ihrer Größe und lange Arrays mit Länge und Summe.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
from sklearn.ensemble import IsolationForest

# Im Browser meldet threadpoolctl beim ersten KMeans eine Warnung der Plattform („JsProxy.as_object_map() is deprecated“).
# Sie betrifft deine Rechnung nicht. Dieser Filter blendet genau diese eine Meldung aus.
warnings.filterwarnings("ignore", message=".*as_object_map.*")

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
zyklen = pd.read_csv("daten/zyklen.csv", dtype={"schicht": SCHICHT_TYP})
FUENF = ["dauer_s", "energie_kwh", "leistung_max_kw", "vibration_mm_s", "kuehlmittel_bar"]
VIER = ["dauer_s", "energie_kwh", "vibration_mm_s", "kuehlmittel_bar"]
print(zyklen.shape, zyklen["maschine"].value_counts().to_dict(), zyklen["datum"].min(), zyklen["datum"].max())


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen.
    Tabellen erscheinen mit ihrer Größe, lange Arrays mit Länge und Summe. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool)):
            return str(w)
        if isinstance(w, dict):
            return "{" + ", ".join(f"{k}: {text(v)}" for k, v in w.items()) + "}"
        if isinstance(w, pd.DataFrame):
            return f"Tabelle {w.shape[0]} × {w.shape[1]}"
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        w = np.ravel(w)
        if len(w) > 12:
            return f"[{len(w)} Werte, Summe {float(np.sum(w)):.6g}]"
        return "[" + " ".join(f"{float(v):.6g}" for v in w) + "]"
    print(*[text(w) for w in werte])

In [ ]:
# Ein Seed für alles: Jede Zelle mit Zufall nimmt random_state=SEED oder np.random.default_rng(SEED + ...).
SEED = 0

### TODO 1 · Messfehler und Lücken
1. `fehlend`: die Zahl der fehlenden Werte je Spalte in `FUENF` und `spindeltemp_c`, als Series. `null_vib`: die Zahl der Zyklen mit einer Schwingung von genau 0,0 mm/s.
2. `meldung_nan`: Passe `KMeans(n_clusters=4, n_init=10, random_state=SEED)` an die fünf Merkmale und die Spindeltemperatur an, fange den `ValueError` mit `try` und `except` ab und speichere die erste Zeile seiner Meldung als Text.
3. `sauber`: `zyklen` mit der Schwingung 0,0 als fehlend gesetzt und ohne die Zeilen mit Lücken in `FUENF`, Index neu gezählt. Die Spindeltemperatur bleibt als Spalte erhalten, sie geht nicht ins Clustering ein.

In [ ]:
# TODO 1a
fehlend = None
null_vib = None

# TODO 1b
meldung_nan = None

# TODO 1c
sauber = None

zeige(None if fehlend is None else fehlend.to_dict(), null_vib, meldung_nan, None if sauber is None else len(sauber))

In [ ]:
check("TODO 1a Lücken und Nullwerte", lambda: list(fehlend.index) == FUENF + ["spindeltemp_c"] and int(fehlend["spindeltemp_c"]) == 32 and int(fehlend.sum()) == 32 and null_vib == 6)
check("TODO 1b Abbruch bei NaN", lambda: isinstance(meldung_nan, str) and "NaN" in meldung_nan)
check("TODO 1c saubere Tabelle", lambda: len(sauber) == 8420 and not sauber[FUENF].isna().any().any() and int((sauber["vibration_mm_s"] == 0).sum()) == 0
      and int(sauber["spindeltemp_c"].isna().sum()) == 32 and list(sauber.index[:3]) == [0, 1, 2])

In [ ]:
# Vorgabe: Ist TODO 1 noch offen, rechnen die folgenden Zellen mit dieser Tabelle weiter. In der Lösung ändert die Zelle nichts.
if sauber is None:
    sauber = zyklen[zyklen["vibration_mm_s"] > 0].reset_index(drop=True)
print(len(sauber))

32 Spindeltemperaturen fehlen, und 6 Zyklen haben eine Schwingung von genau 0,0 mm/s. `KMeans` bricht mit „Input X contains NaN.“ ab. Ohne die 6 Messfehler bleiben 8.420 Zyklen. Die Zeilen ohne Spindeltemperatur behältst du: Das Merkmal bleibt aus dem Clustering heraus, es trennt die Gruppen kaum (Skript Abschnitt 2.6).

### TODO 2 · Abstände von Hand, roh und standardisiert
Das Skript vergleicht Z-00072 mit zwei anderen Zyklen. Hier nimmst du Z-00130, Z-00133 und Z-00149 mit allen fünf Merkmalen, die Zelle zeigt sie als `drei5`.
1. `roh_abst`: ein Array mit den euklidischen Abständen von Z-00130 zu Z-00133 und zu Z-00149 in Rohwerten. `naechster_roh`: die Kennung des näheren der beiden als Text.
2. `skalierer5`: ein `StandardScaler`, angepasst mit `sauber[FUENF]`. `Z5`: die standardisierten fünf Merkmale aller Zyklen in `sauber`.
3. `z_abst` und `naechster_z`: dasselbe wie in Schritt 1, nach dem Standardisieren mit `skalierer5`.

In [ ]:
drei5 = zyklen.set_index("zyklus_id").loc[["Z-00130", "Z-00133", "Z-00149"], FUENF]
print(drei5)

# TODO 2a
roh_abst = None
naechster_roh = None

# TODO 2b
skalierer5 = None
Z5 = None

# TODO 2c
z_abst = None
naechster_z = None

zeige(roh_abst, naechster_roh, z_abst, naechster_z)

In [ ]:
check("TODO 2a Abstände roh", lambda: np.allclose(roh_abst, [32.7718, 79.0018], atol=1e-3) and naechster_roh == "Z-00133")
check("TODO 2b Skalierer auf sauber", lambda: skalierer5.n_samples_seen_ == 8420 and Z5.shape == (8420, 5) and np.allclose(Z5.mean(axis=0), 0, atol=1e-9) and np.allclose(Z5.std(axis=0), 1))
check("TODO 2c Abstände standardisiert", lambda: np.allclose(z_abst, [4.4170, 0.5264], atol=5e-4) and naechster_z == "Z-00149")

In [ ]:
# Vorgabe: Ist TODO 2 noch offen, rechnen die folgenden Zellen mit diesem Skalierer weiter. In der Lösung ändert die Zelle nichts.
if Z5 is None:
    skalierer5 = StandardScaler().fit(sauber[FUENF])
    Z5 = skalierer5.transform(sauber[FUENF])
print(Z5.shape)

Roh liegt Z-00133 mit 32,77 näher als Z-00149 mit 79,0, die Sekunden der Dauer bestimmen den Abstand. Standardisiert dreht es sich um, 4,42 gegen 0,53: Z-00133 arbeitet mit 37,7 bar Kühlmitteldruck, Z-00130 und Z-00149 mit 6,2 und 5,9 bar.

## 2 · k-Means und die Zahl der Gruppen
*Lerneinheit 2 · Skript Abschnitt 2*

k-Means sucht $k$ Mittelpunkte, sodass die Inertia $W = \sum_j \sum_{i \in C_j} \lVert x_i - \mu_j \rVert^2$ möglichst klein wird. Der Lloyd-Algorithmus ordnet abwechselnd zu und mittelt und endet in einem lokalen Optimum, `n_init` startet mehrmals und behält das beste. Die Silhouette $s(i) = (b - a)/\max(a, b)$ bewertet, wie gut die Gruppen getrennt sind. Zum Vergleich rechnet die nächste Zelle die Lösung des Skripts mit vier Merkmalen: `skalierer`, `Z` und `km`.

In [ ]:
skalierer = StandardScaler().fit(sauber[VIER])
Z = skalierer.transform(sauber[VIER])
km = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Z)
print(round(km.inertia_, 1), np.bincount(km.labels_), round(silhouette_score(Z, km.labels_, sample_size=2000, random_state=SEED), 3))

### TODO 3 · Ein Lloyd-Schritt von Hand
Das Skript rechnet zwei Runden an sechs Punkten. Hier rechnest du einen Schritt an allen Zyklen. Die Startpunkte sind die Zeilen 0, 100, 200 und 300 von `Z5`, sie stehen in `start3`.
1. `zuordnung3`: für jede Zeile von `Z5` die Nummer des nächsten Startpunkts. `summe_start3`: die Summe der quadrierten Abstände zu diesem Startpunkt.
2. `mitte3`: die vier neuen Mittelpunkte, jeweils das Mittel der zugeordneten Zeilen, als Array mit 4 Zeilen und 5 Spalten. `summe_mitte3`: die Summe der quadrierten Abstände zu den neuen Mittelpunkten bei gleicher Zuordnung.
3. `km_schritt`: `KMeans(n_clusters=4, init=start3, n_init=1, max_iter=1, random_state=SEED)`, angepasst mit `Z5`.
4. `km_ende`: dasselbe ohne `max_iter`, also bis zum Ende. `km_besser`: `KMeans(n_clusters=4, n_init=10, random_state=SEED)` mit zehn Starts nach k-means++.

In [ ]:
start3 = Z5[[0, 100, 200, 300]]

# TODO 3a
zuordnung3 = None
summe_start3 = None

# TODO 3b
mitte3 = None
summe_mitte3 = None

# TODO 3c
km_schritt = None

# TODO 3d
km_ende = None
km_besser = None

zeige(None if zuordnung3 is None else np.bincount(zuordnung3), summe_start3, summe_mitte3, None if km_ende is None else km_ende.n_iter_,
      None if km_ende is None else km_ende.inertia_, None if km_besser is None else km_besser.inertia_)

In [ ]:
check("TODO 3a Zuordnung zu den Startpunkten", lambda: np.bincount(zuordnung3, minlength=4).tolist() == [5068, 1371, 606, 1375] and math.isclose(summe_start3, 21074.769, abs_tol=0.01))
check("TODO 3b neue Mittelpunkte", lambda: mitte3.shape == (4, 5) and math.isclose(summe_mitte3, 16904.571, abs_tol=0.01))
check("TODO 3c ein Schritt wie KMeans", lambda: np.allclose(km_schritt.cluster_centers_, mitte3, atol=1e-9))
check("TODO 3d lokales und besseres Optimum", lambda: km_ende.n_iter_ == 7 and math.isclose(km_ende.inertia_, 13194.36, abs_tol=0.05) and math.isclose(km_besser.inertia_, 4208.5, abs_tol=0.5))

Nach der ersten Zuordnung gehören zu den Startpunkten 5.068, 1.371, 606 und 1.375 Zyklen. Das Mitteln senkt die Summe von 21.074,8 auf 16.904,6, `KMeans` mit `max_iter=1` findet dieselben Mittelpunkte. Bis zum Ende braucht Lloyd 7 Runden und bleibt bei einer Inertia von 13.194,4 stehen, einem lokalen Optimum. Zehn Starts nach k-means++ kommen auf 4.208,5.

### TODO 4 · Silhouette für mehrere k und die Wahl
1. `wahl5`: ein Wörterbuch mit den Schlüsseln 2 bis 8 und je einem Paar aus Inertia und mittlerer Silhouette von `KMeans(n_clusters=k, n_init=10, random_state=SEED)` auf `Z5`. Die Silhouette rechnest du mit `sample_size=2000, random_state=SEED`, über alle 8.420 Zyklen dauert sie im Browser zu lange.
2. `k_wahl`: das $k$ mit der höchsten Silhouette. `km_z5`: das Modell mit `k_wahl` Gruppen, genauso angepasst.

In [ ]:
# TODO 4a
wahl5 = None

# TODO 4b
k_wahl = None
km_z5 = None

zeige(wahl5, k_wahl)

In [ ]:
check("TODO 4a Inertia und Silhouette", lambda: list(wahl5) == list(range(2, 9))
      and np.allclose([wahl5[k_][0] for k_ in range(2, 9)], [19019.4, 9260.9, 4208.5, 3430.5, 2886.3, 2408.0, 2041.0], rtol=1e-3)
      and np.allclose([wahl5[k_][1] for k_ in range(2, 9)], [0.587, 0.700, 0.729, 0.664, 0.479, 0.479, 0.460], atol=0.03))
check("TODO 4b Wahl von k", lambda: k_wahl == 4 and km_z5.n_clusters == 4 and math.isclose(km_z5.inertia_, 4208.5, abs_tol=0.5))

In [ ]:
# Vorgabe: Ist TODO 4 noch offen, rechnen die folgenden Zellen mit vier Gruppen weiter. In der Lösung ändert die Zelle nichts.
if km_z5 is None:
    km_z5 = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Z5)
print(km_z5.n_clusters)

In [ ]:
if wahl5 is None:
    print("⏳ Erst TODO 4 lösen.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.plot(list(wahl5), [w_[0] for w_ in wahl5.values()], "o-")
    ax1.set_xlabel("Zahl der Gruppen k")
    ax1.set_ylabel("Inertia")
    ax2.plot(list(wahl5), [w_[1] for w_ in wahl5.values()], "o-")
    ax2.set_xlabel("Zahl der Gruppen k")
    ax2.set_ylabel("mittlere Silhouette")
    plt.show()

Die Silhouette steigt von 0,587 bei zwei Gruppen auf 0,729 bei vier und fällt dann auf 0,664 und unter 0,48. Mit fünf Merkmalen liegt sie bei vier Gruppen etwas höher als im Skript mit 0,713, die Inertia knickt wieder bei vier.

### TODO 5 · Profiltabelle in Werkseinheiten
1. `profil5`: eine Tabelle mit den Mittelpunkten von `km_z5`, mit `skalierer5.inverse_transform` in Werkseinheiten zurückgerechnet, den Spalten `FUENF` und einer Spalte `zyklen` mit der Zahl der Zyklen je Gruppe, nach `dauer_s` sortiert.
2. `beschreibung5`: eine Liste mit einem Satz je Gruppe, in der Reihenfolge von `profil5`.

In [ ]:
# TODO 5a
profil5 = None

# TODO 5b
beschreibung5 = None

print(None if profil5 is None else profil5.round(2).to_string())

In [ ]:
check("TODO 5a Profile in Werkseinheiten", lambda: list(profil5.columns[:5]) == FUENF and int(profil5["zyklen"].sum()) == 8420
      and np.allclose(profil5.sort_values("dauer_s")[FUENF].to_numpy(), [[422.09, 0.876, 14.761, 2.020, 5.964], [719.47, 2.158, 23.314, 3.146, 6.474],
                                                                           [760.23, 1.473, 16.862, 1.748, 37.939], [1078.69, 4.017, 29.425, 3.719, 6.968]], rtol=2e-3)
      and profil5.sort_values("dauer_s")["zyklen"].tolist() == [4473, 2344, 700, 903])
check("TODO 5b ein Satz je Gruppe", lambda: len(beschreibung5) == 4 and all(isinstance(s_, str) and len(s_.split()) >= 5 for s_ in beschreibung5))

Die vier Gruppen sind fast dieselben wie im Skript, mit 4.473, 2.344, 700 und 903 Zyklen. Die kurzen dauern im Mittel 422 s bei 14,8 kW Spitzenleistung, die mittleren 719 s bei 23,3 kW, die Gruppe mit 37,9 bar Kühlmitteldruck 760 s bei 16,9 kW und die langen 1.079 s bei 29,4 kW. Die Spitzenleistung steigt mit der Energie, sie bringt wenig Neues.

### TODO 6 · Roh gegen skaliert als Kreuztabelle
1. `km_roh5`: `KMeans(n_clusters=4, n_init=10, random_state=SEED)` auf den Rohwerten `sauber[FUENF]`. `kreuz6`: die Kreuztabelle der Gruppen von `km_roh5` (Zeilen) gegen `km_z5` (Spalten).
2. `ari_roh`: der bereinigte Rand-Index zwischen beiden Einteilungen. `sil_roh5`: die Silhouette der Rohlösung in den Rohwerten, mit `sample_size=2000, random_state=SEED`.

In [ ]:
# TODO 6a
km_roh5 = None
kreuz6 = None

# TODO 6b
ari_roh = None
sil_roh5 = None

print(kreuz6)
zeige(ari_roh, sil_roh5)

In [ ]:
check("TODO 6a Kreuztabelle", lambda: int(kreuz6.to_numpy().sum()) == 8420 and kreuz6.shape == (4, 4) and int((kreuz6.to_numpy() > 100).sum()) == 6)
check("TODO 6b Rand-Index und Silhouette roh", lambda: abs(ari_roh - 0.874) <= 0.01 and abs(sil_roh5 - 0.747) <= 0.025)

Der Rand-Index ist 0,873. Kurze und lange Zyklen findet k-Means auch roh. Die Gruppen um 719 s und 760 s werden nach Dauer gemischt: Eine Rohgruppe enthält 1.645 und 168 Zyklen der beiden, eine andere 692 und 530. Die Silhouette der Rohlösung ist mit 0,747 trotzdem höher als die 0,729 der skalierten, denn sie misst die Trennung fast nur in Sekunden.

### TODO 7 · Stabilität über Starts und Bootstrap
1. `ari_start7`: ein Wörterbuch mit den Schlüsseln 4 und 5. Darin steht je eine Liste mit dem bereinigten Rand-Index zwischen dem ersten und jedem der neun weiteren von zehn Läufen `KMeans(n_clusters=k, n_init=1, random_state=SEED + s)` mit s = 0 bis 9 auf `Z5`.
2. `ari_boot7`: ebenso, mit je fünf Werten. Für jedes k legst du `zufall7 = np.random.default_rng(SEED + 1)` neu an, ziehst fünf Bootstrap-Stichproben der Zeilen von `Z5` mit Zurücklegen, passt darauf `KMeans(n_clusters=k, n_init=10, random_state=SEED)` an und vergleichst `predict(Z5)` mit der Lösung aus zehn Starts auf allen Zyklen.

In [ ]:
# TODO 7a
ari_start7 = None

# TODO 7b
ari_boot7 = None

zeige(None if ari_start7 is None else {k_: [min(v_), np.mean(v_)] for k_, v_ in ari_start7.items()}, None if ari_boot7 is None else {k_: min(v_) for k_, v_ in ari_boot7.items()})

In [ ]:
check("TODO 7a einzelne Starts", lambda: len(ari_start7[4]) == len(ari_start7[5]) == 9 and np.mean(ari_start7[4]) > 0.9 and min(ari_start7[5]) < 0.9)
check("TODO 7b Bootstrap", lambda: len(ari_boot7[4]) == len(ari_boot7[5]) == 5 and min(ari_boot7[4]) > 0.998 and 0.9 < min(ari_boot7[5]) < 1)

Mit vier Gruppen liefert jeder einzelne Start dieselbe Einteilung, auf den Bootstrap-Stichproben bleibt der Rand-Index über 0,999. Mit fünf Gruppen sinkt er bei einzelnen Starts bis 0,58, im Mittel liegt er bei 0,74. Mit zehn Starts bleibt er auf den Stichproben bei mindestens 0,92.

## 3 · Hierarchisch und nach Dichte
*Lerneinheit 3 · Skript Abschnitt 3*

Agglomeratives Clustering verschmilzt Schritt für Schritt die beiden nächsten Gruppen, Ward die beiden, deren Vereinigung die Summe der quadrierten Abstände am wenigsten erhöht. Das Dendrogramm zeigt alle Stufen. Es braucht die Abstände aller Paare, deshalb nimmt es hier die 20 Maschinenprofile. DBSCAN sucht dichte Gebiete mit dem Radius `eps` und der Mindestzahl `min_samples` und gibt dem Rest die Nummer `-1`, Rauschen.

Die nächste Zelle baut die Maschinenprofile wie im Skript: `profile` mit dem Median der Schwingung und der Lagertemperatur über der Halle bis zum Sensortausch an Linie 2, `P` standardisiert, `ward` die Verschmelzungen.

In [ ]:
sensoren = pd.read_csv("daten/sensordaten.csv", dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
produktion = pd.read_csv("daten/produktion.csv", dtype={"schicht": SCHICHT_TYP})
daten = sensoren.merge(maschinen[["maschine", "linie", "typ"]], on="maschine", validate="many_to_one")
daten = daten.merge(produktion[["datum", "schicht", "linie", "hallentemp_c"]], on=["datum", "schicht", "linie"], how="left", validate="many_to_one")
vor = daten[daten["datum"] < "2026-03-02"].assign(temp_halle=lambda d: d["lagertemp_c"] - d["hallentemp_c"])
profile = vor.groupby("maschine").agg(vib=("vibration_mm_s", "median"), temp_halle=("temp_halle", "median"), typ=("typ", "first"))
P = StandardScaler().fit_transform(profile[["vib", "temp_halle"]])
ward = linkage(P, method="ward")
print(ward[-2:, 2].round(3), sorted(np.bincount(fcluster(ward, t=4, criterion="maxclust"))[1:].tolist(), reverse=True))

### TODO 8 · Dendrogramm mit der Stromaufnahme
Das Skript beschreibt die Maschinen mit zwei Werten. Hier kommt der Median der Stromaufnahme `strom_pct` dazu, ebenfalls bis zum 1. März 2026.
1. `profile3`: wie `profile`, mit einer weiteren Spalte `strom`. `P3`: die drei Spalten `vib`, `temp_halle` und `strom`, standardisiert.
2. `ward3`: `linkage(P3, method="ward")`. `gruppen3`: der Schnitt bei vier Gruppen mit `fcluster`. `ari_strom`: der bereinigte Rand-Index zwischen `gruppen3` und dem Schnitt von `ward` bei vier Gruppen.
3. Zeichne das Dendrogramm von `ward3` mit den Maschinennummern als Beschriftung.

In [ ]:
# TODO 8a
profile3 = None
P3 = None

# TODO 8b
ward3 = None
gruppen3 = None
ari_strom = None

# TODO 8c
fig, ax = plt.subplots(figsize=(8, 4))

zeige(None if ward3 is None else ward3[-3:, 2], None if gruppen3 is None else sorted(np.bincount(gruppen3)[1:].tolist()), ari_strom)
plt.show()

In [ ]:
check("TODO 8a Profile mit Strom", lambda: list(profile3.index) == list(profile.index) and math.isclose(float(profile3["strom"].sum()), 1765.7, abs_tol=1e-6) and np.allclose(P3.std(axis=0), 1))
check("TODO 8b Ward mit Strom", lambda: np.allclose(ward3[-3:, 2], [3.7116, 5.7599, 6.3700], atol=1e-3) and sorted(np.bincount(gruppen3)[1:].tolist()) == [1, 5, 6, 8]
      and list(profile3.index[gruppen3 == gruppen3[list(profile3.index).index("M-10")]]) == ["M-10"] and abs(ari_strom - 0.2833) <= 1e-3)

Die letzten drei Verschmelzungen liegen bei 3,71, 5,76 und 6,37. Bei vier Gruppen stehen Montage, Prüfung und Verpackung zusammen (6 Maschinen), die drei Fräszentren mit Entgratstation, Schweißzellen, Laser und Abkantpresse (8), Drehmaschinen, Schleifmaschine und Waschanlagen (5) und der Härteofen allein. Der Rand-Index zur Lösung ohne Strom ist nur 0,28. Ein drittes Merkmal ordnet die Maschinen neu, obwohl die Mediane der Stromaufnahme nur zwischen 86,9 und 89,9 % liegen: Standardisiert zählen auch kleine Unterschiede voll.

Zum Vergleich rechnet die nächste Zelle DBSCAN aus dem Skript mit vier Merkmalen, `dichte`.

In [ ]:
dichte = DBSCAN(eps=0.5, min_samples=10).fit(Z)
print(dichte.labels_.max() + 1, int((dichte.labels_ == -1).sum()))

### TODO 9 · DBSCAN mit k-Abstands-Grafik
1. `k_abstand5`: für jede Zeile von `Z5` der Abstand zum zehnten Nachbarn, der Punkt selbst mitgezählt, aufsteigend sortiert. Tipp: `NearestNeighbors(n_neighbors=10)` und `kneighbors`. Zeichne die Kurve.
2. `eps5`: ein Radius, den du aus der Kurve abliest. `dichte5`: `DBSCAN(eps=eps5, min_samples=10)`, angepasst mit `Z5`.
3. `warm5`: eine boolesche Maske für die Zyklen der kleinsten Gruppe von `dichte5`, ohne Rauschen. `warm5_median`: die Mediane dieser Zyklen in `FUENF` und `spindeltemp_c` als Series.

In [ ]:
# TODO 9a
k_abstand5 = None
fig, ax = plt.subplots(figsize=(7, 4))

# TODO 9b
eps5 = None
dichte5 = None

# TODO 9c
warm5 = None
warm5_median = None

zeige(None if k_abstand5 is None else np.quantile(k_abstand5, 0.99), eps5, None if dichte5 is None else np.bincount(dichte5.labels_ + 1))
print(warm5_median)
plt.show()

In [ ]:
check("TODO 9a k-Abstände", lambda: len(k_abstand5) == 8420 and abs(float(np.quantile(k_abstand5, 0.99)) - 0.5758) <= 1e-3 and bool(np.all(np.diff(k_abstand5) >= 0)))
check("TODO 9b DBSCAN", lambda: dichte5.min_samples == 10 and 0.4 <= dichte5.eps <= 0.7 and dichte5.labels_.max() + 1 == 5 and 40 <= int((dichte5.labels_ == -1).sum()) <= 110)
check("TODO 9c kleinste Gruppe", lambda: int(warm5.sum()) == 22 and np.allclose(warm5_median.to_numpy(), [886.5, 0.505, 6.3, 0.875, 0.2, 31.3], atol=1e-6))

In [ ]:
# Vorgabe: Ist TODO 9 noch offen, rechnen die folgenden Zellen mit diesem DBSCAN weiter. In der Lösung ändert die Zelle nichts.
if dichte5 is None:
    dichte5 = DBSCAN(eps=0.5, min_samples=10).fit(Z5)
if warm5 is None:
    warm5 = dichte5.labels_ == np.argmin(np.bincount(dichte5.labels_[dichte5.labels_ >= 0]))
print(int(warm5.sum()))

Bei 99 % der Zyklen liegt der zehnte Nachbar höchstens 0,58 entfernt, mit fünf Merkmalen etwas weiter als im Skript. Mit `eps=0.5` findet DBSCAN fünf Gruppen und 67 Rauschpunkte, 13 mehr als im Skript. Die kleinste Gruppe sind wieder die 22 Zyklen mit dem Median 886,5 s, 0,505 kWh, nur 6,3 kW Spitzenleistung, 0,2 bar und 31,3 °C Spindeltemperatur.

## 4 · Hauptkomponenten: wenige Richtungen
*Lerneinheit 4 · Skript Abschnitt 4*

Die PCA dreht das Koordinatensystem so, dass die erste Achse die größte Streuung trägt und jede weitere die größte der übrigen, senkrecht zu den vorigen. `explained_variance_ratio_` nennt die Varianzanteile, `components_` die Ladungen, ihre Vorzeichen sind willkürlich. Merkmale in verschiedenen Einheiten gehören vorher standardisiert. Die nächste Zelle rechnet `pca` aus dem Skript mit vier Merkmalen.

In [ ]:
pca = PCA().fit(Z)
print(pca.explained_variance_ratio_.round(3))

### TODO 10 · PCA mit fünf Merkmalen
1. `pca5`: `PCA()`, angepasst mit `Z5`. `anteil5`: die erklärten Varianzanteile, `kumuliert5`: ihre kumulierte Summe.
2. `ladung5`: eine Tabelle mit den Ladungen der ersten beiden Komponenten, Zeilen `PC1` und `PC2`, Spalten `FUENF`.
3. `pca_roh5`: `PCA()` auf den Rohwerten `sauber[FUENF]`.

In [ ]:
# TODO 10a
pca5 = None
anteil5 = None
kumuliert5 = None

# TODO 10b
ladung5 = None

# TODO 10c
pca_roh5 = None

zeige(anteil5, kumuliert5, None if pca_roh5 is None else pca_roh5.explained_variance_ratio_[0])
print(None if ladung5 is None else ladung5.round(3))

In [ ]:
check("TODO 10a Varianzanteile", lambda: np.allclose(anteil5, [0.696463, 0.231425, 0.056762, 0.011920, 0.003430], atol=2e-5) and math.isclose(kumuliert5[1], 0.927888, abs_tol=2e-5))
check("TODO 10b Ladungen, Vorzeichen egal", lambda: list(ladung5.columns) == FUENF
      and np.allclose(np.abs(ladung5.to_numpy()), [[0.4952, 0.5233, 0.5234, 0.4548, 0.0120], [0.2897, 0.0479, 0.0509, 0.2881, 0.9100]], atol=1e-3))
check("TODO 10c PCA der Rohwerte", lambda: abs(pca_roh5.explained_variance_ratio_[0] - 0.998394) <= 1e-5)

In [ ]:
if pca5 is None:
    print("⏳ Erst TODO 10 lösen.")
else:
    werte10 = pca5.transform(Z5)[:, :2]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(werte10[:, 0], werte10[:, 1], c=km_z5.labels_, s=4, cmap="tab10", rasterized=True)
    ax.scatter(werte10[warm5, 0], werte10[warm5, 1], s=30, facecolors="none", edgecolors="black", label="kleine DBSCAN-Gruppe")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend()
    plt.show()

Die erste Komponente trägt 69,6 % der Varianz, die zweite 23,1 %, zusammen 92,8 %. PC1 lädt mit 0,495, 0,523, 0,523 und 0,455 fast gleich auf Dauer, Energie, Spitzenleistung und Schwingung, sie misst die Größe des Zyklus. PC2 lädt mit 0,91 auf den Kühlmitteldruck. Die Spitzenleistung macht PC1 stärker und ändert das Bild kaum. Roh trägt die erste Komponente 99,84 %, sie ist wieder nur die Dauer.

## 5 · Ausreißer finden
*Lerneinheit 5 · Skript Abschnitt 5*

Ein Zyklus mit 4 mm/s Schwingung ist im langen Programm normal, im kurzen auffällig. Deshalb suchst du Ausreißer je Gruppe. Der Mahalanobis-Abstand $d^2(x) = (x - \mu)^\top S^{-1} (x - \mu)$ berücksichtigt die Korrelationen, die Minimum Covariance Determinant schätzt $\mu$ und $S$ robust aus dem dichtesten Kern. Bei ungefähr normalverteilten Gruppen folgt $d^2$ einer $\chi^2$-Verteilung mit so vielen Freiheitsgraden wie Merkmalen. Ob markierte Zyklen mit Problemen zu tun haben, prüfst du an `stoerungen.csv`, immer gegen die Basisrate.

Die nächste Zelle lädt die Störungen wie im Skript: `PASSEND` nennt die fünf Meldungen, die zu Zyklen passen, `meldung` hat eine Zeile je Schicht und Maschine mit mindestens einer solchen Meldung.

In [ ]:
stoerungen = pd.read_csv("daten/stoerungen.csv", dtype={"schicht": SCHICHT_TYP})
PASSEND = ["Werkzeugbruch", "Spindel läuft unrund", "Kühlmittelpumpe undicht", "Rohteile mit Übermaß angeliefert", "Falsches Programm gewählt"]
meldung = stoerungen[stoerungen["beschreibung"].isin(PASSEND)][["datum", "schicht", "maschine"]].drop_duplicates().assign(meldung=1)
print(len(stoerungen), len(meldung))

### TODO 11 · Robuste Abstände je Gruppe
1. `grenze5`: das 0,999-Quantil der $\chi^2$-Verteilung mit 5 Freiheitsgraden. Tipp: `stats.chi2.ppf`.
2. `robust5`: für jeden Zyklus der robuste quadrierte Mahalanobis-Abstand in seiner Gruppe von `km_z5`, mit `MinCovDet(random_state=SEED)` je Gruppe auf `Z5` angepasst. `markiert5`: die Maske der Zyklen über `grenze5`. `je_gruppe5`: die Zahl der markierten Zyklen je Gruppe als Array.
3. `global5`: wie viele Zyklen eine einzige `MinCovDet(random_state=SEED)` über alle Zyklen zusammen markiert.

In [ ]:
# TODO 11a
grenze5 = None

# TODO 11b
robust5 = None
markiert5 = None
je_gruppe5 = None

# TODO 11c
global5 = None

zeige(grenze5, None if markiert5 is None else int(markiert5.sum()), je_gruppe5, global5)

In [ ]:
check("TODO 11a Grenze", lambda: math.isclose(grenze5, 20.515, abs_tol=1e-3))
check("TODO 11b robuste Abstände je Gruppe", lambda: len(robust5) == 8420 and bool(np.array_equal(markiert5, robust5 > grenze5)) and 150 <= int(markiert5.sum()) <= 170
      and np.allclose(sorted(je_gruppe5.tolist()), [5, 29.5, 40, 85], atol=3))
check("TODO 11c MCD über alle Zyklen", lambda: abs(global5 - 4030) <= 40)

In [ ]:
# Vorgabe: Ist TODO 11 noch offen, rechnet TODO 12 mit diesen markierten Zyklen. In der Lösung ändert die Zelle nichts.
if markiert5 is None:
    markiert5 = np.zeros(len(Z5), dtype=bool)
    for g_ in range(km_z5.n_clusters):
        teil_ = km_z5.labels_ == g_
        markiert5[teil_] = MinCovDet(random_state=SEED).fit(Z5[teil_]).mahalanobis(Z5[teil_]) > stats.chi2.ppf(0.999, df=5)
print(int(markiert5.sum()))

Die Grenze liegt bei 20,52. Je Gruppe markiert die robuste Rechnung 159 Zyklen, 85 unter den kurzen, 40 unter den mittleren, 29 unter den langen und 5 unter den Kühlmittel-Zyklen. Eine MCD über alle Zyklen zusammen markiert 4.030. Sie schätzt Mittelpunkt und Streuung aus der dichteren Hälfte der Zyklen, und die kurzen allein sind schon 53 %.

### TODO 12 · Abgleich mit den Meldungen gegen die Basisrate
1. `in_schicht`: für jeden Zyklus in `sauber` eine 1, wenn seine Schicht an seiner Maschine eine passende Meldung hat, sonst 0, als Array. Verknüpfe dazu über `datum`, `schicht` und `maschine` mit `meldung`. `basisrate`: der Anteil aller Zyklen mit 1.
2. `iso5`: die Maske der Zyklen, die `IsolationForest(contamination=0.02, random_state=SEED)` auf `Z5` als Ausreißer markiert.
3. `anteile12`: ein Wörterbuch mit den Schlüsseln `"alle"`, `"Isolation Forest"`, `"MCD je Gruppe"` und `"MCD ohne kleine Gruppe"` und dem Anteil der jeweiligen Zyklen in Schichten mit passender Meldung. „Ohne kleine Gruppe“ heißt `markiert5` ohne die Zyklen in `warm5`.

In [ ]:
# TODO 12a
in_schicht = None
basisrate = None

# TODO 12b
iso5 = None

# TODO 12c
anteile12 = None

zeige(basisrate, None if iso5 is None else int(iso5.sum()), anteile12)

In [ ]:
check("TODO 12a Basisrate", lambda: len(in_schicht) == 8420 and math.isclose(basisrate, 0.102613, abs_tol=1e-5))
check("TODO 12b Isolation Forest", lambda: len(iso5) == 8420 and int(iso5.sum()) == 169)
check("TODO 12c Anteile gegen die Basisrate", lambda: math.isclose(anteile12["alle"], 0.102613, abs_tol=1e-5) and 0.15 <= anteile12["Isolation Forest"] <= 0.40
      and abs(anteile12["MCD je Gruppe"] - 0.734) <= 0.03 and abs(anteile12["MCD ohne kleine Gruppe"] - 0.844) <= 0.03 and anteile12["MCD je Gruppe"] > 2 * anteile12["Isolation Forest"])

10,3 % aller Zyklen liegen in einer Schicht mit passender Meldung. Von den 169 Zyklen des Isolation Forest sind es 24,3 %, von den 159 der robusten Abstände je Gruppe 73,6 %, ohne die 22 Zyklen der kleinen Gruppe 84,7 %. Mit fünf Merkmalen trifft die Rechnung je Gruppe etwas öfter als im Skript, der Wald seltener. Ein Anteil ohne Basisrate sagt nichts: Mit irgendeiner Meldung in der Schicht wären es schon für alle Zyklen 41,2 %.

## 6 · Gruppen prüfen ohne Zielgröße
*Lerneinheit 6 · Skript Abschnitt 6*

Ohne Zielgröße bleiben innere Güte, Stabilität, Deutbarkeit und wenige Etiketten. Ab dem 14. September trägt der Leitstand das Programm ein. Diese Etiketten dienen einmal als äußere Prüfung, nachdem $k$ ohne sie gewählt ist. Der bereinigte Rand-Index vergleicht zwei Einteilungen unabhängig von der Nummerierung der Gruppen.

### TODO 13 · Äußere Prüfung mit den Wochen ab 14. September
`mit13` markiert die Zyklen mit Programm, `vor13` die Zyklen vor dem 14. September. Die Zahl der Gruppen hast du in TODO 4 ohne die Programme gewählt.
1. `kreuz13`: die Kreuztabelle der Programme (Zeilen) gegen die Gruppen von `km_z5` (Spalten) für die Zyklen mit Programm. `ari13`: der bereinigte Rand-Index dazu.
2. Die strenge Fassung: `skalierer_vor`, ein `StandardScaler` nur mit den Zyklen vor dem 14. September, und `km_vor`, `KMeans(n_clusters=km_z5.n_clusters, n_init=10, random_state=SEED)` auf diesen Zyklen, standardisiert mit `skalierer_vor`. `streng13`: der bereinigte Rand-Index zwischen den Programmen und den Gruppen, die `km_vor.predict` den Zyklen mit Programm zuordnet.

In [ ]:
mit13 = sauber["programm"].notna().to_numpy()
vor13 = (sauber["datum"] < "2026-09-14").to_numpy()

# TODO 13a
kreuz13 = None
ari13 = None

# TODO 13b
skalierer_vor = None
km_vor = None
streng13 = None

print(kreuz13)
zeige(int(mit13.sum()), ari13, streng13)

In [ ]:
check("TODO 13a Programme gegen Gruppen", lambda: int(kreuz13.to_numpy().sum()) == 1866 and abs(ari13 - 0.990) <= 0.005
      and kreuz13.loc["Warmlauf"].idxmax() == kreuz13.loc["Lagerdeckel"].idxmax() and int(kreuz13.loc["Warmlauf"].max()) == 6)
check("TODO 13b nur mit den Wochen davor angepasst", lambda: skalierer_vor.n_samples_seen_ == 6554 and km_vor.n_clusters == 4 and abs(streng13 - 0.990) <= 0.005)

1.866 Zyklen haben ein Programm. Die vier Gruppen treffen die Programme fast genau, der Rand-Index ist 0,990. Alle 6 als „Warmlauf“ beschrifteten Zyklen stecken in der Gruppe der Lagerdeckel, dazu 2 Zyklen „Gehäuse klein“. Skalierer und Gruppen nur aus den Wochen davor ergeben denselben Rand-Index von 0,990.

## 7 · Grenzen des unüberwachten Lernens
*Lerneinheit 7 · Skript Abschnitt 7*

k-Means liefert immer $k$ Gruppen, auch in Daten ohne Struktur, und die stärkste Struktur ist nicht immer die gesuchte. Die beiden Bonusaufgaben zeigen das an Rauschen und probieren mit HDBSCAN ein Verfahren ohne `eps`.

### Bonus 1 · Gruppen in dreidimensionalem Rauschen
1. `rauschen3`: 1.000 Punkte, gleichverteilt im Einheitswürfel, mit `np.random.default_rng(SEED).uniform(size=(1000, 3))`.
2. `sil_rauschen`: ein Wörterbuch mit den Schlüsseln 2 bis 6 und der mittleren Silhouette von `KMeans(n_clusters=k, n_init=10, random_state=SEED)` auf `rauschen3`, über alle 1.000 Punkte.

In [ ]:
# Bonus 1
rauschen3 = None
sil_rauschen = None

zeige(sil_rauschen)

In [ ]:
check("Bonus 1 Silhouette in Rauschen", lambda: rauschen3.shape == (1000, 3) and list(sil_rauschen) == [2, 3, 4, 5, 6]
      and np.allclose(list(sil_rauschen.values()), [0.251, 0.252, 0.277, 0.279, 0.296], atol=0.03) and sil_rauschen[6] > sil_rauschen[2])

In gleichverteiltem Rauschen liegt die Silhouette zwischen 0,25 und 0,30, am höchsten bei sechs Gruppen. k-Means schneidet den Würfel in Stücke. Bei den Zyklen sind es 0,73 bei vier Gruppen. Erst der Vergleich mit Daten ohne Struktur ordnet eine Silhouette ein.

### Bonus 2 · HDBSCAN ohne eps
1. `hdb5`: `HDBSCAN(min_cluster_size=15, copy=True)`, angepasst mit `Z5`. `copy=True` lässt `Z5` unverändert, neuere Versionen von scikit-learn warnen ohne diese Angabe.
2. `kreuz_hdb`: die Kreuztabelle der Gruppen von `hdb5` (Zeilen) gegen `dichte5` (Spalten). Sieh sie dir mit `print(kreuz_hdb)` an. Einzelne Zyklen am Rand können im Browser und in Colab anders fallen.

In [ ]:
# Bonus 2
hdb5 = None
kreuz_hdb = None

zeige(None if hdb5 is None else hdb5.labels_.max() + 1, None if hdb5 is None else min(np.bincount(hdb5.labels_[hdb5.labels_ >= 0])),
      None if hdb5 is None else round(adjusted_rand_score(dichte5.labels_, hdb5.labels_), 2))

In [ ]:
check("Bonus 2 HDBSCAN", lambda: hdb5.labels_.max() + 1 == 5 and 50 <= int((hdb5.labels_ == -1).sum()) <= 66 and 22 in np.bincount(hdb5.labels_[hdb5.labels_ >= 0]).tolist()
      and int(kreuz_hdb.to_numpy().sum()) == 8420)

HDBSCAN findet ohne `eps` dieselben fünf Gruppen, darunter wieder die 22 Zyklen, und etwa 58 Rauschpunkte. Mit DBSCAN stimmt die Einteilung bis auf einige Zyklen am Rand überein, der Rand-Index ist gerundet 1,0.

## Zum Nachdenken

1. In TODO 6 ist die Silhouette der Rohlösung höher als die der skalierten. Welche Lösung gibst du dem Werk, und wie begründest du das ohne Etiketten?
2. Nach TODO 7 sind fünf Gruppen mit zehn Starts stabil. Reicht das als Beleg für fünf Programme?
3. Die robusten Abstände je Gruppe treffen Schichten mit Meldung viel öfter als der Isolation Forest. Welche Liste gibst du der Instandhaltung, und was sagst du dazu?
4. Ein Kollege schlägt vor, die Zahl der Gruppen künftig so zu wählen, dass der Rand-Index mit den Programmen am höchsten ist. Was antwortest du?
5. In TODO 8 ordnet die Stromaufnahme die Maschinen neu, obwohl ihre Mediane nur wenige Prozentpunkte auseinanderliegen. Was folgt daraus für die Wahl der Merkmale?

<details><summary>Hinweise zu den Antworten</summary>

1. Die skalierte. Die Rohsilhouette misst fast nur die Dauer in Sekunden, und die Profile zeigen, dass roh zwei Programme mit verschiedenem Kühlmitteldruck gemischt werden. Silhouetten lassen sich nur bei gleichen Merkmalen in gleicher Skalierung vergleichen, die Deutbarkeit der Profile entscheidet mit.
2. Nein. Stabil heißt nur, dass das Verfahren bei kleinen Änderungen dasselbe findet. Mit fünf Gruppen hängt das Ergebnis einzelner Starts vom Zufall ab, die Silhouette ist niedriger, und im Skript teilt die fünfte Gruppe ein Programm nach Maschine.
3. Die Liste je Gruppe, mit dem Hinweis, dass ein Treffer nur heißt, dass in derselben Schicht eine passende Meldung vorliegt. Die 22 Zyklen der kleinen Gruppe gehören gesondert betrachtet, sie sind selten, aber kaum in Schichten mit Meldung.
4. Dann sind die Programme die Zielgröße, und das ist überwachtes Lernen. Das ist erlaubt, braucht dann aber neue Prüfdaten. Die Wochen ab 14. September wären als Prüfung verbraucht.
5. Standardisieren gibt jedem Merkmal dasselbe Gewicht, auch einem mit kleiner Streuung und ohne klare Gruppen. Welche Merkmale in den Abstand eingehen, ist eine Entscheidung über Ähnlichkeit und gehört begründet.
</details>

## Wie es weitergeht

Du hast Lücken und Messfehler geprüft, Abstände und einen Lloyd-Schritt von Hand gerechnet, die Zahl der Gruppen mit der Silhouette gewählt, Profile gelesen, Stabilität geprüft, Maschinen hierarchisch und Zyklen nach Dichte gruppiert, mit PCA verdichtet, Ausreißer je Gruppe gefunden und gegen die Basisrate geprüft. Offen bleibt die Zeit: Clustering sieht jeden Zyklus für sich, den Sägezahn der Schwingung innerhalb eines Programms sieht es nicht. **Modul 11 „Zeitreihen und Forecasting“** fragt: Was passiert morgen, und wann läuft etwas aus dem Ruder?